# 15 · 手撸 KV-Cache：用空间换时间

> **学习目标**：自回归生成时为什么要 KV-cache？手实现「无 cache」和「有 cache」两个版本，验证输出**逐元素相等**，并画出延迟随长度增长的曲线。
>
> **预备**：07 + 14 已过。
>
> **为什么重要**：vLLM / TensorRT-LLM 等推理框架的核心优化全围绕 KV-cache。理解它的形状、生长方式、显存代价，才能估算「我这个模型能并发多少 user」。

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
import math, time
import matplotlib.pyplot as plt

torch.manual_seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

## 1. 为什么需要 KV cache —— 浪费在哪

**自回归生成的流程**：
```
step 1: 输入 [t1]                                -> 算 K[1], V[1], attn -> token 2
step 2: 输入 [t1, t2]                            -> 算 K[1..2], V[1..2], attn -> token 3
step 3: 输入 [t1, t2, t3]                        -> 算 K[1..3], V[1..3], attn -> token 4
...
step L: 输入 [t1..tL]                            -> 算 K[1..L], V[1..L], attn -> token L+1
```

**痛点**：step L 时，`K[1..L-1]` 与 `V[1..L-1]` **完全可以直接复用**，没必要再算一遍。

**KV-cache**：把每步算出的 K、V 存起来，下一步只算新 token 那一行的 K/V，然后 append。

**计算量**：
- 无 cache：每步 O(L · d)（要重算所有 K/V）+ O(L²) attention
- 有 cache：每步 O(d)（只算 1 个新 K/V）+ O(L) attention（Q 只 1 行）

**生成 L 个 token 的总开销**：O(L²)（有 cache） vs O(L³)（无 cache）—— L=1000 时差三个量级。

## 2. 两个版本的 CausalAttention

为了对比清楚，写一个 module 同时支持两种 forward 模式：

In [ ]:
class CausalAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.proj = nn.Linear(d_model, d_model, bias=False)

    # -------- 模式 A：无 cache，每次都对整段输入算 --------
    def forward_full(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, L, D) -> (B, L, D)"""
        B, L, D = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        q = q.view(B, L, self.n_heads, self.d_head).transpose(1, 2)
        k = k.view(B, L, self.n_heads, self.d_head).transpose(1, 2)
        v = v.view(B, L, self.n_heads, self.d_head).transpose(1, 2)
        scores = q @ k.transpose(-2, -1) / math.sqrt(self.d_head)
        mask = torch.triu(torch.ones(L, L, device=x.device, dtype=torch.bool), diagonal=1)
        scores = scores.masked_fill(mask, float('-inf'))
        attn = scores.softmax(-1)
        out = (attn @ v).transpose(1, 2).contiguous().view(B, L, D)
        return self.proj(out)

    # -------- 模式 B：有 cache，只算新 token --------
    def forward_step(self, x_new: torch.Tensor, cache: dict | None = None):
        """
        x_new: (B, 1, D)  当前 step 只输入新 1 个 token
        cache: {'k': (B, H, T, dh), 'v': (B, H, T, dh)} 或 None（第一步）
        返回: out (B, 1, D), new_cache
        """
        B, _, D = x_new.shape
        q, k, v = self.qkv(x_new).chunk(3, dim=-1)
        q = q.view(B, 1, self.n_heads, self.d_head).transpose(1, 2)   # (B, H, 1, dh)
        k = k.view(B, 1, self.n_heads, self.d_head).transpose(1, 2)
        v = v.view(B, 1, self.n_heads, self.d_head).transpose(1, 2)

        if cache is None:
            k_all, v_all = k, v
        else:
            k_all = torch.cat([cache['k'], k], dim=2)                  # 沿 L 维 append
            v_all = torch.cat([cache['v'], v], dim=2)

        # attention：q 只有 1 行（最新 token 当 query），看所有历史 K/V
        scores = q @ k_all.transpose(-2, -1) / math.sqrt(self.d_head)   # (B, H, 1, T+1)
        # 这里不需要 causal mask！因为 q 是最新一行，它本来就「看不到未来」（未来还没生成）
        attn = scores.softmax(-1)
        out = (attn @ v_all).transpose(1, 2).contiguous().view(B, 1, D)
        return self.proj(out), {'k': k_all, 'v': v_all}

# Smoke
attn = CausalAttention(d_model=32, n_heads=4).to(device).eval()
x = torch.randn(2, 8, 32, device=device)
y_full = attn.forward_full(x)
print('forward_full :', x.shape, '->', y_full.shape)

cache = None
outs = []
for t in range(8):
    o, cache = attn.forward_step(x[:, t:t+1], cache)
    outs.append(o)
y_step = torch.cat(outs, dim=1)
print('forward_step :', y_step.shape, '   cache K shape:', cache['k'].shape)

## 3. 关键正确性测试：两种模式输出**逐元素相等**

**这是判断 KV-cache 实现是否正确的唯一标准**。差一个 epsilon 都说明哪里抄错了。

In [ ]:
torch.manual_seed(123)
attn = CausalAttention(d_model=64, n_heads=8).to(device).eval()
B, L, D = 2, 16, 64
x = torch.randn(B, L, D, device=device)

with torch.no_grad():
    y_full = attn.forward_full(x)

    cache = None
    outs = []
    for t in range(L):
        o, cache = attn.forward_step(x[:, t:t+1], cache)
        outs.append(o)
    y_step = torch.cat(outs, dim=1)

diff = (y_full - y_step).abs().max().item()
print(f'最大逐元素差: {diff:.3e}')
assert torch.allclose(y_full, y_step, atol=1e-5), 'KV-cache 实现错了！'
print('✅ 两种模式输出一致 —— KV-cache 实现正确')

## 4. 延迟曲线 —— 无 cache O(L²) vs 有 cache O(L)

**测量协议**：
- 序列从 16 长到 512，每个 L 测「生成下一个 token」的耗时
- 无 cache：每步都对 `[0..L]` 全段重算 attention
- 有 cache：用 `forward_step` + 累积 cache

**预期**：无 cache 应该随 L 显著上扬（attention 矩阵 L×L 主导）；有 cache 几乎水平。

In [ ]:
torch.manual_seed(0)
attn = CausalAttention(d_model=256, n_heads=8).to(device).eval()
B = 1

Ls = [16, 32, 64, 128, 256, 512]
results = {'L': [], 'no_cache_ms': [], 'cache_ms': []}

for L in Ls:
    x = torch.randn(B, L, 256, device=device)

    # warm-up（避免首次 CUDA kernel 编译影响）
    for _ in range(2):
        with torch.no_grad():
            _ = attn.forward_full(x)
    if device == 'cuda':
        torch.cuda.synchronize()

    # 无 cache：再算一次「再加 1 个 token」 = 对 L+1 段重新算 attention（取最后 1 行）
    x_plus = torch.randn(B, L + 1, 256, device=device)
    t0 = time.perf_counter()
    for _ in range(10):
        with torch.no_grad():
            _ = attn.forward_full(x_plus)
    if device == 'cuda':
        torch.cuda.synchronize()
    no_cache_ms = (time.perf_counter() - t0) / 10 * 1000

    # 有 cache：先建好 cache 到长度 L，然后只跑 1 step
    cache = None
    with torch.no_grad():
        for t in range(L):
            _, cache = attn.forward_step(x[:, t:t+1], cache)
    if device == 'cuda':
        torch.cuda.synchronize()
    x_new = torch.randn(B, 1, 256, device=device)
    t0 = time.perf_counter()
    for _ in range(10):
        with torch.no_grad():
            _, _ = attn.forward_step(x_new, cache)
    if device == 'cuda':
        torch.cuda.synchronize()
    cache_ms = (time.perf_counter() - t0) / 10 * 1000

    results['L'].append(L); results['no_cache_ms'].append(no_cache_ms); results['cache_ms'].append(cache_ms)
    print(f'L={L:4d}  无 cache {no_cache_ms:7.2f} ms   有 cache {cache_ms:7.2f} ms   省 {(1-cache_ms/no_cache_ms)*100:5.1f}%')

In [ ]:
plt.figure(figsize=(9, 4))
plt.plot(results['L'], results['no_cache_ms'], '-o', label='无 cache（每步重算全部）')
plt.plot(results['L'], results['cache_ms'],    '-o', label='有 cache（每步只算 1 新 token）')
plt.xlabel('已生成序列长度 L'); plt.ylabel('生成下一个 token 的时间 (ms)')
plt.title(f'KV-cache 收益（device={device}）')
plt.legend(); plt.grid(True)
plt.show()
print('→ 无 cache 应该明显往上弯（接近线性，attention 部分 L²）')
print('→ 有 cache 应该几乎水平 —— 每步开销与 L 几乎无关')

## 5. 显存代价 —— KV cache 多大？

**公式**：
$$\text{KV cache size} = 2 \times B \times L \times H_{kv} \times d_{head} \times \text{bytes\_per\_value}$$

其中 2 是 K + V 两份，bytes\_per\_value = 2 (fp16/bf16) 或 4 (fp32)。

**举例**（fp16）：
- Llama-2 7B：H_kv = 32, d_head = 128, batch=1, L=2048 → 2 × 32 × 2048 × 128 × 2 = **32 MB / user**
- Llama-3 8B（GQA）：H_kv = 8, d_head = 128, batch=1, L=8192 → 2 × 8 × 8192 × 128 × 2 = **32 MB / user**
  （GQA 让 KV head 数砍 4 倍，于是同显存可跑 4 倍长上下文！）

In [ ]:
def kv_cache_size_mb(B, L, n_kv_heads, d_head, bytes_per_value=2):
    return 2 * B * L * n_kv_heads * d_head * bytes_per_value / 1024 / 1024

print(f'{"模型":<22} {"H_kv":>5} {"d_h":>4} {"B":>3} {"L":>6} {"KV cache":>12}')
print('-' * 60)
for name, hkv, dh, B, L in [
    ('Llama-2 7B',        32, 128, 1,  2048),
    ('Llama-2 7B',        32, 128, 4,  2048),
    ('Llama-2 7B',        32, 128, 1,  8192),
    ('Llama-3 8B (GQA)',   8, 128, 1,  8192),
    ('Llama-3 8B (GQA)',   8, 128, 1, 32768),
    ('Qwen2.5 7B (GQA)',   4, 128, 1, 32768),
]:
    mb = kv_cache_size_mb(B, L, hkv, dh)
    print(f'{name:<22} {hkv:>5} {dh:>4} {B:>3} {L:>6} {mb:>10.1f} MB')

print('\n→ 关键洞察：')
print('  1. KV cache 随 batch × seq_len 线性涨。多用户高并发 = 显存吃紧。')
print('  2. GQA 直接砍 KV head 数 → 同显存可跑更长上下文 / 更多用户。')
print('  3. 这就是为什么 vLLM 的 PagedAttention（按页管理 KV）这么重要。')

In [ ]:
# 一个常见服务化场景：你的 8GB GPU 跑 Llama-3 8B (fp16)，能并发多少 user？
# 假设模型本身 ~ 16 GB（fp16），那已经塞不下。换 INT4 量化模型 ~ 5 GB。
MODEL_MB = 5 * 1024     # INT4 量化后的模型权重大小
GPU_MB   = 8 * 1024     # 8GB GPU
OS_RESERVED = 1 * 1024  # OS / framework 占用预留
BUDGET_FOR_KV = GPU_MB - MODEL_MB - OS_RESERVED
print(f'KV cache 可用预算: {BUDGET_FOR_KV} MB')

per_user_at_L = lambda L: kv_cache_size_mb(B=1, L=L, n_kv_heads=8, d_head=128)
for L in [2048, 4096, 8192, 16384, 32768]:
    per_u = per_user_at_L(L)
    max_u = BUDGET_FOR_KV // per_u
    print(f'  上下文 {L:>5} → 每 user {per_u:6.1f} MB → 并发上限 {int(max_u):>3} user')
print('\n→ 这就是「我的卡能并发几个 user」的标准估算法。生产里再乘 0.6 留余量。')

## 深入思考

1. **为什么 forward_step 不需要 causal mask 了？**
   - Q 只有 1 行（最新 token），它看 K/V 全部历史 —— 全部历史按定义都是「过去」，不会偷看未来。无需 mask。
2. **KV cache 形状为什么是 `(B, n_kv_heads, L, d_head)` 而不是 `(B, L, n_kv_heads, d_head)`？**
   - 计算时 `q @ k.transpose(-2, -1)` 用的就是这种 head-first 布局。提前 transpose 让推理时无需再 transpose，省时间。
3. **多个 user 并发时 KV cache 怎么管理？**
   - 每个 user 一份独立 cache。**问题**：固定 max_len 的 cache 浪费——短对话也占 32768。**vLLM PagedAttention**：把 cache 按页（block）切，按需分配，碎片化变低 → 同样显存能塞更多 user。
4. **KV cache 能不能压缩？**
   - 能。MQA、GQA 是「砍 head 数」；KV cache quantization（FP8 / INT8 cache）是「砍 bit 数」；MLA（DeepSeek-V2）是「低秩压缩」。组合可省 4~8 倍。
5. **首 token 延迟（TTFT）和 KV cache 关系？**
   - 首 token 时 cache 还没建（叫 "prefill" 阶段），要把 prompt 全部跑一遍。**prefill 是吃算力的、decode 是吃显存带宽的** —— 两种典型瓶颈不同。

改一改：把 `forward_step` 里的 `torch.cat` 换成预分配的固定大小 cache（`cache[:, :, t]`），看延迟有没有进一步降低。

## 自检 ✅

- [ ] 解释「KV cache 用空间换什么」（省的是 K/V 重算，更省的是 K @ Q 的左乘 O(L)）。
- [ ] 默写 KV cache 大小公式。
- [ ] 给一个 8GB GPU、Llama-3 8B INT4、L=8192，能口算并发 user 数量。
- [ ] 解释「为什么 GQA 让长上下文成为可能」。
- [ ] 解释 prefill 与 decode 两个阶段的瓶颈差异。

## Stage 3 + Stage 4 全部完成 🎉

→ 回到 [00-基础-Foundations/README.md](../../README.md) 把 Stage 3 + 4 的 checkbox 全部打勾
→ 准备进入 [01-RAG](../../../01-RAG/) 或 [04-模型微调-Finetuning](../../../04-模型微调-Finetuning/)

**走到这里你已经具备**：
- ✅ 从 char 到 token 到 logits 全链路心智模型
- ✅ 训练循环 + 混合精度 + warmup + grad clip
- ✅ 手撸 self-attention 与 nn.MultiheadAttention 对齐
- ✅ 现代 LLM 4 大组件：RMSNorm / SwiGLU / GQA / RoPE
- ✅ KV-cache 实现与显存估算

**应当能做到**：
- 看懂 HuggingFace `modeling_llama.py` 整个文件
- 看懂任意一篇 Transformer 改进论文的实现部分
- 估算「我手上的 GPU 能 host 多大模型、能并发几个用户」